In [1]:
import os
import zipfile
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
zip_path = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("Exists:", os.path.exists(zip_path))

Exists: True


In [5]:
extract_dir = "/content/dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

Unzipped to: /content/dataset


In [6]:
for root, dirs, files in os.walk(extract_dir):
    if "scenario23" in root.lower():
        print(root)

/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/mmWave_data
/content/dataset/scenario23_dev/unit1/GPS_data
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final
/content/dataset/scenario23_dev/unit2
/content/dataset/scenario23_dev/unit2/pitch
/content/dataset/scenario23_dev/unit2/roll
/content/dataset/scenario23_dev/unit2/distance
/content/dataset/scenario23_dev/unit2/x_speed
/content/dataset/scenario23_dev/unit2/GPS_data
/content/dataset/scenario23_dev/unit2/y_speed
/content/dataset/scenario23_dev/unit2/speed
/content/dataset/scenario23_dev/unit2/z_speed
/content/dataset/scenario23_dev/unit2/altitude
/content/dataset/scenario23_dev/unit2/height


In [8]:
author_dir = "/content/Image beam"
os.makedirs(author_dir, exist_ok=True)


print(author_dir)

/content/Image beam


In [9]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"

In [10]:
DATA_ROOT = "/content/dataset/scenario23_dev"   # update if needed
print("Exists:", os.path.exists(DATA_ROOT))

Exists: True


In [11]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"  # adjust path

needed = [
    "build_net.py",
    "data_feed.py",
    "main_beam.py",
    "main_beam_eval.py",
    "scenario23_img_beam_train.csv",
    "scenario23_img_beam_val.csv",
    "scenario23_img_beam_test.csv",
]

for f in needed:
    p = os.path.join(AUTHOR_ROOT, f)
    print(f, "->", os.path.exists(p), p)

build_net.py -> True /content/drive/MyDrive/Image beam/build_net.py
data_feed.py -> True /content/drive/MyDrive/Image beam/data_feed.py
main_beam.py -> True /content/drive/MyDrive/Image beam/main_beam.py
main_beam_eval.py -> True /content/drive/MyDrive/Image beam/main_beam_eval.py
scenario23_img_beam_train.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_train.csv
scenario23_img_beam_val.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_val.csv
scenario23_img_beam_test.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_test.csv


In [12]:
train_csv = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv")
df_img_author = pd.read_csv(train_csv)

print(df_img_author.head())
print(df_img_author.columns.tolist())

   index                                          unit1_rgb  unit1_beam
0   3532  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
1   2224  ../scenario23_dev/unit1/camera_data/image_BS1_...          14
2   9416  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
3   8510  ../scenario23_dev/unit1/camera_data/image_BS1_...          20
4   6877  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
['index', 'unit1_rgb', 'unit1_beam']


In [13]:
sample_paths = df_img_author.iloc[:10, 1].tolist()
print(sample_paths)

for p in sample_paths[:5]:
    print("RAW:", p, "EXISTS:", os.path.exists(str(p)))

['../scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg', '../scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg']
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg EXISTS: False
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg EXISTS: False
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg EXISTS: False
RAW: ../scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg EXISTS: 

In [14]:
from pathlib import Path

image_map = {}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_map[f] = os.path.join(root, f)

print("Indexed images:", len(image_map))
print(list(image_map.items())[:5])

Indexed images: 11387
[('image_BS1_379_16_59_31.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_379_16_59_31.jpg'), ('image_BS1_3857_17_09_15.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3857_17_09_15.jpg'), ('image_BS1_2719_17_05_56.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2719_17_05_56.jpg'), ('image_BS1_10364_17_56_56.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10364_17_56_56.jpg'), ('image_BS1_5298_17_14_32.jpg', '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_5298_17_14_32.jpg')]


In [15]:
def fix_img_csv_paths(csv_path, out_path):
    dfc = pd.read_csv(csv_path).copy()

    # author code takes row.values[1:3], so column index 1 must be image path
    path_col = dfc.columns[1]

    dfc[path_col] = dfc[path_col].apply(lambda x: image_map.get(os.path.basename(str(x).strip()), str(x)))
    dfc.to_csv(out_path, index=False)

    print("Saved:", out_path)
    print(dfc.head())
    return dfc

fixed_train = fix_img_csv_paths(
    os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv"),
    "/content/scenario23_img_beam_train_fixed.csv"
)

fixed_val = fix_img_csv_paths(
    os.path.join(AUTHOR_ROOT, "scenario23_img_beam_val.csv"),
    "/content/scenario23_img_beam_val_fixed.csv"
)

fixed_test = fix_img_csv_paths(
    os.path.join(AUTHOR_ROOT, "scenario23_img_beam_test.csv"),
    "/content/scenario23_img_beam_test_fixed.csv"
)

Saved: /content/scenario23_img_beam_train_fixed.csv
   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
Saved: /content/scenario23_img_beam_val_fixed.csv
   index                                          unit1_rgb  unit1_beam
0   1542  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   6743  /content/dataset/scenario23_dev/unit1/camera_d...          10
2  10999  /content/dataset/scenario23_dev/unit1/camera_d...           2
3   5992  /content/dataset/scenario23_dev/unit1/camera_d...          15
4   7698  /content/dataset/scenario23_dev/unit1/camera_d...           7
Saved: /content/scenario23_img_bea

In [16]:
for p in fixed_train.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg True


In [18]:
from skimage import io
from torch.utils.data import Dataset, DataLoader


def create_samples(root):
    f = pd.read_csv(root)
    data_samples = []
    for idx, row in f.iterrows():
        img_paths = row.values[1:3]
        data_samples.append(img_paths)
    return data_samples

class DataFeedAuthor(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = root_dir
        self.samples = create_samples(self.root)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = io.imread(sample[0])
        if self.transform:
            img = self.transform(img)
        label = int(sample[1])
        return img, label

In [19]:
import torchvision.transforms as transf

img_resize = transf.Resize((224, 224))
img_norm = transf.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

proc_pipe = transf.Compose([
    transf.ToPILImage(),
    img_resize,
    transf.ToTensor(),
    img_norm
])

In [21]:
batch_size = 64
val_batch_size = 64

train_loader_img = DataLoader(
    DataFeedAuthor("/content/scenario23_img_beam_train_fixed.csv", transform=proc_pipe),
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader_img = DataLoader(
    DataFeedAuthor("/content/scenario23_img_beam_val_fixed.csv", transform=proc_pipe),
    batch_size=val_batch_size,
    shuffle=True, num_workers=2
)

test_loader_img = DataLoader(
    DataFeedAuthor("/content/scenario23_img_beam_test_fixed.csv", transform=proc_pipe),
    batch_size=val_batch_size,
    shuffle=True, num_workers=2
)

In [22]:
imgs, labels = next(iter(train_loader_img))
print(imgs.shape, labels.shape)
print(labels[:10])

torch.Size([64, 3, 224, 224]) torch.Size([64])
tensor([17,  4,  2, 26, 15, 20, 15, 30, 11, 17])


In [23]:

from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model_img = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model_img.fc = nn.Linear(model_img.fc.in_features, 64)   # author code uses 64
model_img = model_img.to(device)

print(model_img.fc)

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 224MB/s]


Linear(in_features=2048, out_features=64, bias=True)


In [24]:
def evaluate_topk(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()

    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)

            for k in ks:
                correct_k = pred[:k].eq(labels.view(1, -1)).sum().item()
                correct[k] += correct_k

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

In [25]:
import torch.optim as optim


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_img.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[4, 8, 12], gamma=0.1)

num_epochs = 20
best_top1 = -1
best_state = None
history = []

for epoch in range(1, num_epochs + 1):
    model_img.train()
    running_loss = 0.0
    total_train = 0

    for imgs, labels in train_loader_img:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model_img(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        running_loss += loss.item() * bs
        total_train += bs

    scheduler.step()

    train_loss = running_loss / total_train
    val_metrics = evaluate_topk(model_img, val_loader_img, device, ks=(1, 2, 3, 5))

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Top1: {val_metrics['top1']:.2f} | "
        f"Top2: {val_metrics['top2']:.2f} | "
        f"Top3: {val_metrics['top3']:.2f} | "
        f"Top5: {val_metrics['top5']:.2f}"
    )

    if val_metrics["top1"] > best_top1:
        best_top1 = val_metrics["top1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model_img.state_dict().items()}
        torch.save(best_state, "/content/best_resnet50_img.pth")
        print("Saved best model")

Epoch 01 | Train Loss: 0.9807 | Top1: 80.68 | Top2: 95.64 | Top3: 98.45 | Top5: 99.24
Saved best model
Epoch 02 | Train Loss: 0.4508 | Top1: 84.02 | Top2: 97.57 | Top3: 98.95 | Top5: 99.53
Saved best model
Epoch 03 | Train Loss: 0.4243 | Top1: 85.42 | Top2: 97.13 | Top3: 98.98 | Top5: 99.33
Saved best model
Epoch 04 | Train Loss: 0.3999 | Top1: 85.57 | Top2: 97.86 | Top3: 99.09 | Top5: 99.68
Saved best model
Epoch 05 | Train Loss: 0.2961 | Top1: 87.88 | Top2: 98.30 | Top3: 99.30 | Top5: 99.62
Saved best model
Epoch 06 | Train Loss: 0.2586 | Top1: 87.53 | Top2: 98.39 | Top3: 99.30 | Top5: 99.62
Epoch 07 | Train Loss: 0.2335 | Top1: 88.00 | Top2: 98.27 | Top3: 99.30 | Top5: 99.53
Saved best model
Epoch 08 | Train Loss: 0.2026 | Top1: 86.62 | Top2: 98.39 | Top3: 99.24 | Top5: 99.56
Epoch 09 | Train Loss: 0.1640 | Top1: 87.85 | Top2: 98.21 | Top3: 99.30 | Top5: 99.62
Epoch 10 | Train Loss: 0.1492 | Top1: 87.91 | Top2: 98.21 | Top3: 99.27 | Top5: 99.65
Epoch 11 | Train Loss: 0.1395 | Top1: 

In [26]:
hist_img = pd.DataFrame(history)
hist_img

,epoch,train_loss,top1,top2,top3,top5
0,1,0.980723,80.679157,95.638173,98.448478,99.238876
1,2,0.450845,84.016393,97.570258,98.946136,99.531616
2,3,0.424326,85.421546,97.131148,98.975410,99.326698
3,4,0.399887,85.567916,97.862998,99.092506,99.677986
4,5,0.296103,87.880562,98.302108,99.297424,99.619438
5,6,0.258616,87.529274,98.389930,99.297424,99.619438
6,7,0.233452,87.997658,98.272834,99.297424,99.531616
7,8,0.202602,86.621780,98.389930,99.238876,99.560890
8,9,0.164027,87.851288,98.214286,99.297424,99.619438
9,10,0.149223,87.909836,98.214286,99.268150,99.648712


In [27]:
best_state = torch.load("/content/best_resnet50_img.pth", map_location=device)
model_img.load_state_dict(best_state)
model_img = model_img.to(device)

test_metrics = evaluate_topk(model_img, test_loader_img, device, ks=(1, 2, 3, 5))
print("Test metrics:", test_metrics)

Test metrics: {'top1': 87.18173836698858, 'top2': 98.06848112379281, 'top3': 99.47322212467077, 'top5': 99.82440737489026}


In [28]:
results_img = pd.DataFrame([{
    "Modality": "Image (ResNet50)",
    "Top-1": test_metrics["top1"],
    "Top-2": test_metrics["top2"],
    "Top-3": test_metrics["top3"],
    "Top-5": test_metrics["top5"],
}])

results_img

,Modality,Top-1,Top-2,Top-3,Top-5
0,Image (ResNet50),87.181738,98.068481,99.473222,99.824407


mlp cell 

In [29]:
AUTHOR_POS_ROOT = "/content/drive/MyDrive/Pos beam"   # change path if needed